In [1]:

import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

from sklearn.dummy import DummyClassifier


In [2]:

# Load Microsoft stock data
df = pd.read_csv("../data/raw/msft.us.txt")

# Convert dates and sort chronologically
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

# Remove unused column
df = df.drop(columns=["OpenInt"])

# Handle zero-volume observations
df["Volume"] = df["Volume"].replace(0, np.nan).ffill()

# Engineer the same four features as Random Forest
df["Daily_Return"] = df["Close"].pct_change()

df["MA_5"] = df["Close"].rolling(5).mean() / df["Close"] - 1
df["MA_20"] = df["Close"].rolling(20).mean() / df["Close"] - 1

df["Volatility_5"] = df["Daily_Return"].rolling(5).std()

# Target: 1 if next trading day's closing price increases
df["Next_Close"] = df["Close"].shift(-1)
df["Target"] = (df["Next_Close"] > df["Close"]).astype(int)

# Remove incomplete observations
df = df.dropna(subset=[
    "Daily_Return",
    "MA_5",
    "MA_20",
    "Volatility_5",
    "Next_Close"
])

# Define model features
features = [
    "Daily_Return",
    "MA_5",
    "MA_20",
    "Volatility_5"
]

X = df[features]
y = df["Target"]

print("Total observations:", len(df))
print("\nTarget distribution:")
print(y.value_counts(normalize=True))


Total observations: 7963

Target distribution:
Target
0    0.538239
1    0.461761
Name: proportion, dtype: float64


In [3]:

# Chronological 80/20 train/test split
split_index = int(len(df) * 0.8)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

# Create a pipeline for scaling and Logistic Regression
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

print("Training observations:", len(X_train))
print("Testing observations:", len(X_test))

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True))

print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True))


Training observations: 6370
Testing observations: 1593

Training target distribution:
Target
0    0.549608
1    0.450392
Name: proportion, dtype: float64

Testing target distribution:
Target
1    0.507219
0    0.492781
Name: proportion, dtype: float64


In [4]:

# Train Logistic Regression model
pipeline.fit(X_train, y_train)

# Generate predictions on unseen test data
y_pred = pipeline.predict(X_test)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)

print("Logistic Regression training complete!")
print(f"Test Accuracy: {accuracy:.4f}")

# Examine predicted class distribution
print("\nPredicted class distribution:")
print(pd.Series(y_pred).value_counts(normalize=True))


Logistic Regression training complete!
Test Accuracy: 0.4940

Predicted class distribution:
0    0.992467
1    0.007533
Name: proportion, dtype: float64


In [5]:

# Evaluate Logistic Regression
print("Logistic Regression Classification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# Compare against baseline
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)

baseline_predictions = baseline.predict(X_test)

print("\nModel Accuracy Comparison:")
print(f"Logistic Regression: {accuracy_score(y_test, y_pred):.4f}")
print(f"Random Forest:       0.4922")
print(f"Majority Baseline:   {accuracy_score(y_test, baseline_predictions):.4f}")


Logistic Regression Classification Report:
              precision    recall  f1-score   support

           0       0.49      0.99      0.66       785
           1       0.58      0.01      0.02       808

    accuracy                           0.49      1593
   macro avg       0.54      0.50      0.34      1593
weighted avg       0.54      0.49      0.33      1593


Confusion Matrix:
[[780   5]
 [801   7]]

Model Accuracy Comparison:
Logistic Regression: 0.4940
Random Forest:       0.4922
Majority Baseline:   0.4928
